---

# UDF (User defined functions)

- PySpark has a lot of useful built-in function but having your own customizable function gives you more flexibility.
- A user defined function, or UDF, is **a Python method that the user writes to perform a specific bit of logic**.
    - Once written, the method is called via the **pyspark.sql.functions.udf()** method. The result **is stored as a variable** and can be called as a normal Spark function.
    - UDF are **similar to traditional RDBMS user defined functions**, which were registered in the database library and can be used as regular functions.
    - In PySpark, you create a function similar to python using **def keyword**, then wrap them in **udf() method** and register them.
- Once registered they can be used on DataFrame or SQL expressions.
- The **default type of udf() is StringType.**

**Uses :**

- It is used to create reusable functions in spark.
- Logic is complex
- No built-in functions exists

**Limitations :**

- UDFs operate row-wise, they are not vectorized like Pandas UDF
- UDF’s are most expensive operations and should be only used when it is extremely essential.
- Slower than in-built functions
- Breaks the **catalyst optimizer**
- Following are the reasons to avoid udf:
    - UDFs are Black box to PySpark, hence you can’t apply optimization and you lose all optimization PySpark does on DataFrame/Dataset.
    - If built-in function available.

### Types of UDF

- PySpark or Python UDF
- Pandas UDF
- Scala UDF

Pandas UDF 

- The Pandas UDF **processes data in batches**, using Pandas' vectorized operations for faster execution on larger datasets.

In [0]:
# Syntax of UDF

def function(param):
    return param

udf_function = udf(function, returnType)
df.withColumn('udf_column_name', udf_function(column))


In [0]:
import pyspark.sql.types as T
import pyspark.sql.functions as F

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, IntegerType

data = [("hello",), ("world",), ("pyspark",)]
df = spark.createDataFrame(data, ["text"])
df.show()

In [0]:
# Define the UDF
def capitalize_text(text):
    text.capitalize()

In [0]:
# Register the UDF
capitalize_udf = udf(capitalize_text, StringType())

In [0]:
capitalize_udf('rohan')

In [0]:
df = df.withColumn("text_capitalized", capitalize_udf(df["text"]))
df.show()

In [0]:
data = [
    ("affenpinscher", 0, 9, 173, 298),
    ("Border_terrier", 73, 127, 341, 335),
    ("Kuvasz", 0, 0, 499, 327),
    ("Great_Pyrenees", 124, 225, 403, 374),
    ("schipperke", 146, 29, 416, 509),
    ("groenendael", 168, 0, 469, 374),
    ("Bedlington_terrier", 10, 12, 462, 332),
    ("Lhasa", 39, 1, 499, 373),
    ("Kerry_blue_terrier", 17, 16, 300, 482),
    ("vizsla", 112, 93, 276, 236),
    ("Eskimo_dog", 43, 20, 472, 461),
    ("cairn", 71, 2, 319, 302),
    ("EntleBucher", 307, 94, 515, 448)
]

df_dog = spark.createDataFrame(data, schema = ["dog_list"])
df_dog.show()

In [0]:
from pyspark.sql.types import StructType, StructField, ArrayType, StringType

schema = StructType([
    StructField("dog_list", ArrayType(StringType()), True)
])

df = spark.createDataFrame([(list(map(str, row)),) for row in data], schema)
df.show(truncate=False)

In [0]:
def dog_parse(doglist):
    dogs = []
    for dog in doglist:
        (breed, start_x, start_y, end_x, end_y) = dog.split(",")
        dogs.append((breed, int(start_x), int(start_y), int(end_x), int(end_y)))
    return dogs

In [0]:
doglist = [
    "affenpinscher,0,9,173,298",
    "Kuvasz,0,0,499,327"
]
dog_parse(doglist=doglist)

In [0]:
df.printSchema()

In [0]:
df_1 = df.withColumn("size", F.size('dog_list'))
df_1.show()